# RelCheck — Master Evaluation Notebook
**CS298 Spring 2026 | Siddhi Patil | SJSU**

This notebook runs the **complete RelCheck pipeline** end-to-end:
- ✅ Pilot demo on 12 images
- ✅ R-Bench 200-image evaluation
- ✅ Two baselines + full RelCheck
- ✅ Ablation study (3 variants)
- ✅ R-CHAIR annotation helper
- ✅ All figures and metric tables

**How to use:** Set your API key and GitHub URL in Section 0, then click **Runtime → Run all**.

---

## Section 0: Setup
Run these cells once at the start of every Colab session.

In [1]:
# ── Mount Google Drive (persistent storage) ──────────────────────────────
# Images (~1.6 GB) and eval results are stored here so they survive
# Colab session resets. You only download images ONCE — ever.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE        = "/content/drive/MyDrive/RelCheck_Data"
DRIVE_IMAGES_DIR  = f"{DRIVE_BASE}/images"
DRIVE_EVAL_DIR    = f"{DRIVE_BASE}/eval"
DRIVE_FIGURES_DIR  = f"{DRIVE_BASE}/figures"
DRIVE_MODELS_DIR   = f"{DRIVE_BASE}/hf_cache"
DRIVE_ANNOT_DIR    = f"{DRIVE_BASE}/annotations"


import os
for d in [DRIVE_BASE, DRIVE_IMAGES_DIR, DRIVE_EVAL_DIR, DRIVE_FIGURES_DIR, DRIVE_MODELS_DIR, DRIVE_ANNOT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Google Drive mounted → {DRIVE_BASE}")
# ── Point HuggingFace cache → Drive (model weights download once, ever) ──
import os
os.environ["HF_HOME"]              = DRIVE_MODELS_DIR
os.environ["TRANSFORMERS_CACHE"]   = DRIVE_MODELS_DIR
os.environ["HUGGINGFACE_HUB_CACHE"]= DRIVE_MODELS_DIR
print(f"   HF model cache  : {DRIVE_MODELS_DIR}")

print(f"   Images cache : {DRIVE_IMAGES_DIR}")
print(f"   Eval outputs : {DRIVE_EVAL_DIR}")
print(f"   Figures      : {DRIVE_FIGURES_DIR}")


Mounted at /content/drive
✅ Google Drive mounted → /content/drive/MyDrive/RelCheck_Data
   Images cache : /content/drive/MyDrive/RelCheck_Data/images
   Eval outputs : /content/drive/MyDrive/RelCheck_Data/eval
   Figures      : /content/drive/MyDrive/RelCheck_Data/figures


In [2]:
# ── Restore eval results from Drive (if resuming a session) ──────────────
# If you've run experiments before, copy existing results back into the repo
# so you don't have to re-run completed sections.
import shutil, glob, os

REPO_DIR = "/content/RelCheck"
restored = 0

# Only restore if repo already exists (clone-repo cell creates it later)
if os.path.exists(f"{REPO_DIR}/relcheck"):
    os.makedirs(f"{REPO_DIR}/eval",    exist_ok=True)
    os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)

    for pattern in [
        f"{DRIVE_EVAL_DIR}/*.csv",
        f"{DRIVE_EVAL_DIR}/*.json",
        f"{DRIVE_FIGURES_DIR}/*.png",
        f"{DRIVE_FIGURES_DIR}/*.pdf",
    ]:
        for f in glob.glob(pattern):
            dest_dir = f"{REPO_DIR}/eval" if DRIVE_EVAL_DIR in f else f"{REPO_DIR}/figures"
            shutil.copy2(f, dest_dir)
            restored += 1

if restored:
    print(f"\u2705 Restored {restored} file(s) from Drive")
    print("   You can skip sections whose CSVs are already present.")
else:
    print("\u2139\ufe0f  No prior results found (repo not yet cloned, or Drive empty) \u2014 fresh run.")

# ── Restore R-Bench annotations from Drive (avoids 5 MB re-download) ──
import zipfile, pathlib
RBENCH_DIR      = "/content/R-Bench"
ANNOTATIONS_DIR = f"{RBENCH_DIR}/data"
ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
DRIVE_ANNOT_ZIP = f"{DRIVE_ANNOT_DIR}/rbench_annotations.zip"
if (not os.path.exists(ANNOTATIONS_DIR) or
        len(list(pathlib.Path(ANNOTATIONS_DIR).rglob('*.json'))) == 0):
    if os.path.exists(DRIVE_ANNOT_ZIP):
        os.makedirs(RBENCH_DIR, exist_ok=True)
        os.makedirs(ANNOTATIONS_DIR, exist_ok=True)
        shutil.copy2(DRIVE_ANNOT_ZIP, ANNOTATIONS_ZIP)
        with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
            z.extractall(ANNOTATIONS_DIR)
        print('✅ R-Bench annotations restored from Drive')
    else:
        print('ℹ️  R-Bench annotations not in Drive yet — will download in Section 4')



ℹ️  No prior results found in Drive — fresh run.


In [3]:
# ── GPU Check ──────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → GPU (T4 or A100)."
    )
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
if vram_gb < 14:
    print("⚠️  Less than 14 GB VRAM detected. Consider switching to A100 in Colab Pro.")

✅ GPU: NVIDIA A100-SXM4-80GB  |  VRAM: 85.1 GB


In [4]:
# ── Install dependencies (run once; ~3 min) ────────────────────────────────
!pip install -q \
    spacy transformers[torch] accelerate bitsandbytes \
    together python-Levenshtein nltk \
    pandas matplotlib seaborn requests tqdm Pillow
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All dependencies installed


In [5]:
# ── Clone your GitHub repo ─────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/siddhipatil503/RelCheck.git"
REPO_DIR = "/content/RelCheck"

import os, sys, shutil

# Check for .git dir (not just directory existence — makedirs can create it without git)
if os.path.exists(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git pull
elif os.path.exists(REPO_DIR):
    # Directory exists but is NOT a git repo (created by earlier makedirs)
    shutil.rmtree(REPO_DIR)
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

# Add relcheck/ package to Python path
sys.path.insert(0, f"{REPO_DIR}/relcheck")

# Create output directories
os.makedirs(f"{REPO_DIR}/eval",    exist_ok=True)
os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)

print(f"\u2705 Repo ready at {REPO_DIR}")


fatal: not a git repository (or any of the parent directories): .git
✅ Repo ready at /content/RelCheck


In [6]:
# ── API Key (Together.ai) ─────────────────────────────────────────────────
# Get a free key at https://api.together.xyz
TOGETHER_API_KEY = "tgp_v1_BUl6LMJYvkKBqL6gQppIFShXiYHoJQhFOmOYJkRuJFQ"  # ← PASTE YOUR KEY HERE (do NOT commit to GitHub)

import os
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
assert TOGETHER_API_KEY, "Together.ai API key is required."
print("✅ Together.ai API key set")


✅ Together.ai API key set


---
## Section 1: Load Models
BLIP-2 is loaded once here and shared across all pipeline stages to avoid OOM errors.
Loading takes ~3–5 minutes on first run (downloading ~15 GB of weights).

In [7]:
# ── Load BLIP-2 (captioner — model being studied) ─────────────────────────
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
import torch

BLIP2_MODEL_ID = "Salesforce/blip2-flan-t5-xl"
print(f"Loading {BLIP2_MODEL_ID} in 8-bit ...")

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
blip2_processor = Blip2Processor.from_pretrained(BLIP2_MODEL_ID)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
blip2_model.eval()
print(f"✅ BLIP-2 loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading Salesforce/blip2-flan-t5-xl in 8-bit ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1289 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ BLIP-2 loaded | VRAM: 4.7 GB


In [8]:
# ── Load LLaVA-1.5-7B (independent verifier — different architecture from BLIP-2) ──
# This is the cross-model verifier that eliminates circular self-verification.
# BLIP-2 generates captions; LLaVA verifies the extracted triples.
# On A100 (40 GB): BLIP-2 ~10 GB + LLaVA ~8 GB = ~18 GB — fits comfortably.
from transformers import LlavaForConditionalGeneration, AutoProcessor

LLAVA_MODEL_ID = "llava-hf/llava-1.5-7b-hf"
print(f"Loading {LLAVA_MODEL_ID} in 8-bit ...")

llava_processor = AutoProcessor.from_pretrained(LLAVA_MODEL_ID)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
llava_model.eval()
print(f"✅ LLaVA-1.5-7B loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"   Total VRAM used (both models): {torch.cuda.memory_allocated()/1e9:.1f} / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

Loading llava-hf/llava-1.5-7b-hf in 8-bit ...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✅ LLaVA-1.5-7B loaded | VRAM: 12.0 GB
   Total VRAM used (both models): 12.0 / 85 GB


In [9]:
# ── Ensure relcheck/ is on Python path (safe to re-run after runtime restart) ─
import sys, os as _os, shutil as _shutil
_repo_dir = "/content/RelCheck"
_relcheck_path = f"{_repo_dir}/relcheck"
if not _os.path.exists(_relcheck_path):
    # Remove stale non-git directory if it exists, then clone fresh
    if _os.path.exists(_repo_dir):
        _shutil.rmtree(_repo_dir)
    _os.system(f"git clone https://github.com/siddhipatil503/RelCheck.git {_repo_dir}")
if _relcheck_path not in sys.path:
    sys.path.insert(0, _relcheck_path)

# ── Patch VQAVerifier to reuse pre-loaded BLIP-2; wire LLaVA into pipeline ─
import relation_verifier as _rv

def _patched_vqa_init(self, model_name=BLIP2_MODEL_ID):
    self.processor = blip2_processor
    self.model     = blip2_model
    self.device    = next(blip2_model.parameters()).device
    print("[VQAVerifier] Using pre-loaded BLIP-2 (shared).")

_rv.VQAVerifier.__init__ = _patched_vqa_init
print("✅ VQAVerifier patched to share BLIP-2")

⚠️  /content/RelCheck/relcheck not found — cloning repo...


ModuleNotFoundError: No module named 'relation_verifier'

In [ ]:
# ── Ensure relcheck/ is on Python path (safe to re-run after runtime restart) ─
import sys, os as _os, shutil as _shutil
_repo_dir = "/content/RelCheck"
_relcheck_path = f"{_repo_dir}/relcheck"
if not _os.path.exists(_relcheck_path):
    # Remove stale non-git directory if it exists, then clone fresh
    if _os.path.exists(_repo_dir):
        _shutil.rmtree(_repo_dir)
    _os.system(f"git clone https://github.com/siddhipatil503/RelCheck.git {_repo_dir}")
if _relcheck_path not in sys.path:
    sys.path.insert(0, _relcheck_path)

# ── Initialize RelCheck Pipeline (with LLaVA verifier + LLM extractor) ────
from relcheck_pipeline import RelCheckPipeline

pipeline = RelCheckPipeline(
    together_api_key  = TOGETHER_API_KEY,
    llava_model       = llava_model,       # cross-model verifier
    llava_processor   = llava_processor,
    use_llm_extractor = True,              # Llama-3.3-70B triple extraction
)
torch.cuda.empty_cache()
print(f"✅ Pipeline ready | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("   Verifier:  LLaVA-1.5-7B (independent of BLIP-2)")
print("   Extractor: Llama-3.3-70B via Together.ai (spaCy fallback)")
print("   Corrector: Llama-3.3-70B via Together.ai")

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────
from PIL import Image

# ── BLIP-2 caption generation ──────────────────────────────────────────────
def generate_caption(image: Image.Image, max_tokens: int = 80) -> str:
    """Generate a BLIP-2 caption (the model whose hallucinations we correct)."""
    inputs = blip2_processor(
        images=image, text="Describe this image in detail.",
        return_tensors="pt"
    ).to(blip2_model.device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=max_tokens)
    return blip2_processor.decode(ids[0], skip_special_tokens=True).strip()

# ── LLaVA: independent R-POPE evaluator ───────────────────────────────────
# All R-POPE metrics use LLaVA as the evaluator — not BLIP-2.
# This eliminates circular evaluation (BLIP-2 grading its own answers).
#
# Three evaluation modes:
#   llava_direct(image, q)             → no caption context (Baseline 1)
#   llava_with_caption(image, caption, q) → caption provided as context (Baseline 2, RelCheck)
#
# For Baseline 1 (No Correction): LLaVA answers directly from the image.
# For Baseline 2 (Self-Refine):   LLaVA gets BLIP-2's self-refined caption as context.
# For RelCheck:                   LLaVA gets RelCheck's corrected caption as context.
# This cleanly isolates the effect of caption quality on downstream VQA performance.

LLAVA_DIRECT_TEMPLATE = (
    "USER: <image>\n{question} Answer with yes or no only. ASSISTANT:"
)
LLAVA_CONTEXT_TEMPLATE = (
    "USER: <image>\nImage description: {caption}\n"
    "{question} Based on the description and the image, answer with yes or no only. ASSISTANT:"
)

def _llava_answer_tokens(prompt: str, image: Image.Image) -> str:
    """Run LLaVA with given prompt+image, return 'yes' or 'no'."""
    inputs = llava_processor(
        text=prompt, images=image, return_tensors="pt"
    ).to(llava_model.device)
    with torch.no_grad():
        ids = llava_model.generate(**inputs, max_new_tokens=5)
    decoded = llava_processor.decode(ids[0], skip_special_tokens=True)
    answer  = decoded.split("ASSISTANT:")[-1].strip().lower()
    return "yes" if answer.startswith("yes") else "no"

def llava_direct(image: Image.Image, question: str) -> str:
    """LLaVA answers R-Bench question directly from image (no caption context)."""
    return _llava_answer_tokens(
        LLAVA_DIRECT_TEMPLATE.format(question=question), image
    )

def llava_with_caption(image: Image.Image, caption: str, question: str) -> str:
    """LLaVA answers R-Bench question given a caption as context."""
    return _llava_answer_tokens(
        LLAVA_CONTEXT_TEMPLATE.format(caption=caption, question=question), image
    )

# ── R-POPE metric computation ──────────────────────────────────────────────
def compute_rpope_metrics(predictions: list[dict]) -> dict:
    """Compute R-POPE metrics from a list of {pred, gt} dicts."""
    tp = sum(1 for p in predictions if p['pred'] == 'yes' and p['gt'] == 'yes')
    tn = sum(1 for p in predictions if p['pred'] == 'no'  and p['gt'] == 'no')
    fp = sum(1 for p in predictions if p['pred'] == 'yes' and p['gt'] == 'no')
    fn = sum(1 for p in predictions if p['pred'] == 'no'  and p['gt'] == 'yes')
    total     = len(predictions)
    accuracy  = (tp + tn) / total if total else 0
    precision = tp / (tp + fp)    if (tp + fp) else 0
    recall    = tp / (tp + fn)    if (tp + fn) else 0
    f1        = (2*precision*recall)/(precision+recall) if (precision+recall) else 0
    yes_ratio = (tp + fp) / total if total else 0
    return dict(accuracy=accuracy, precision=precision, recall=recall,
                f1=f1, yes_ratio=yes_ratio, tp=tp, tn=tn, fp=fp, fn=fn, total=total)

print("✅ Helper functions ready")
print("   R-POPE evaluator: LLaVA-1.5-7B (independent of BLIP-2)")

---
## Section 2: Pilot Demo (12 Images)
Runs the full pipeline on the 12 images in `images/`.
This is a qualitative sanity check and produces `eval/pilot_results.csv`.

In [ ]:
import glob
IMAGES_DIR  = f"{REPO_DIR}/images"
image_paths = sorted(
    glob.glob(f"{IMAGES_DIR}/*.jpeg") +
    glob.glob(f"{IMAGES_DIR}/*.jpg")  +
    glob.glob(f"{IMAGES_DIR}/*.webp") +
    glob.glob(f"{IMAGES_DIR}/*.png")
)
print(f"Found {len(image_paths)} pilot images")

PILOT_CSV = f"{REPO_DIR}/eval/pilot_results.csv"
pilot_results = pipeline.run_batch(
    image_paths      = image_paths,
    blip2_processor  = blip2_processor,
    blip2_model      = blip2_model,
    output_csv       = PILOT_CSV,
)
print(f"\n✅ Pilot complete! {len(pilot_results)} images processed.")
print(f"Results saved → {PILOT_CSV}")

In [ ]:
import pandas as pd
df_pilot = pd.read_csv(PILOT_CSV)

print("=" * 80)
print("PILOT RESULTS SUMMARY")
print("=" * 80)
for _, row in df_pilot.iterrows():
    fname = row['image_path'].split('/')[-1]
    changed = row['original_caption'] != row['corrected_caption']
    print(f"\n[{fname}]")
    print(f"  ORIGINAL:  {row['original_caption']}")
    if changed:
        print(f"  CORRECTED: {row['corrected_caption']}")
    else:
        print(f"  (no correction needed)")
    print(f"  triples={row['n_triples']}, hallucinated={row['n_hallucinated']}, edit_rate={row['edit_rate']:.3f}")

print("\n" + "=" * 80)
print(f"Total hallucinations detected: {df_pilot['n_hallucinated'].sum()} "
      f"across {df_pilot['n_triples'].sum()} triples")
print(f"Images with ≥1 hallucination: {df_pilot['any_hallucination'].sum()}/{len(df_pilot)}")
print(f"Avg edit rate: {df_pilot['edit_rate'].mean():.3f}")

---
## Section 3: R-Bench Setup (200-Image Subset)
Downloads R-Bench annotations and a 200-image subset of nocaps validation images.
**This only needs to run once** — results are cached in `/content/rbench/`.

In [ ]:
import os, pathlib, zipfile

RBENCH_DIR = "/content/R-Bench"

# ── Step 1: Clone repo (for eval scripts) ─────────────────────────────────
if not os.path.exists(RBENCH_DIR):
    !git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}
    print("✅ R-Bench repo cloned")
else:
    print("✅ R-Bench repo already present")

# ── Step 2: Download annotation files from Google Drive ───────────────────
# The JSON annotation files are NOT in the GitHub repo — they're on Google Drive.
# Link: https://drive.google.com/file/d/1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH/view
# This is only ~5 MB (just the annotation JSONs, NOT the 1.6 GB images folder).
# Images for our 200-image subset are downloaded individually in a later cell.

GDRIVE_FILE_ID = "1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH"
ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
ANNOTATIONS_DIR = f"{RBENCH_DIR}/data"

# ── Restore annotations from Drive if already downloaded before ──────────
DRIVE_ANNOT_ZIP = f"{DRIVE_ANNOT_DIR}/rbench_annotations.zip"
if not (os.path.exists(ANNOTATIONS_DIR) and len(list(pathlib.Path(ANNOTATIONS_DIR).rglob("*.json"))) > 0):
    if os.path.exists(DRIVE_ANNOT_ZIP):
        print("♻️  Restoring R-Bench annotations from Drive...")
        os.makedirs(ANNOTATIONS_DIR, exist_ok=True)
        import shutil
        shutil.copy2(DRIVE_ANNOT_ZIP, ANNOTATIONS_ZIP)
        with zipfile.ZipFile(ANNOTATIONS_ZIP, "r") as z:
            z.extractall(ANNOTATIONS_DIR)
        print("✅ Annotations restored from Drive (no download needed)")

if not os.path.exists(ANNOTATIONS_DIR) or len(list(pathlib.Path(ANNOTATIONS_DIR).rglob("*.json"))) == 0:
    print("Downloading R-Bench annotations from Google Drive (~5 MB)...")
    !pip install -q gdown
    !gdown --id {GDRIVE_FILE_ID} -O {ANNOTATIONS_ZIP}

    if os.path.exists(ANNOTATIONS_ZIP):
        os.makedirs(ANNOTATIONS_DIR, exist_ok=True)
        try:
            with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
                z.extractall(ANNOTATIONS_DIR)
            print(f"✅ Annotations extracted to {ANNOTATIONS_DIR}")
        import shutil as _sh
        _sh.copy2(ANNOTATIONS_ZIP, DRIVE_ANNOT_DIR)
        print(f"💾 Annotations backed up to Drive")
        except zipfile.BadZipFile:
            # Might be a tar.gz or raw folder — try alternative approaches
            print("Not a standard zip. Trying gdown with --fuzzy flag...")
            os.remove(ANNOTATIONS_ZIP)
            !gdown "https://drive.google.com/file/d/{GDRIVE_FILE_ID}/view?usp=sharing" \
                -O {ANNOTATIONS_ZIP} --fuzzy
            if os.path.exists(ANNOTATIONS_ZIP):
                try:
                    with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
                        z.extractall(ANNOTATIONS_DIR)
                    print(f"✅ Annotations extracted (2nd attempt)")
                except:
                    print("⚠️  Still failed. See manual download instructions below.")
else:
    print("✅ R-Bench annotations already present")

# NOTE: We do NOT download the full 1.6 GB images folder here.
# Only the 200 images we need are fetched individually in the download-images cell.

# ── Verify what we got ────────────────────────────────────────────────────
json_files = sorted(pathlib.Path(RBENCH_DIR).rglob("*.json"))
print(f"\nFound {len(json_files)} JSON files:")
for f in json_files[:20]:
    size_kb = f.stat().st_size // 1024
    print(f"  {f.relative_to(RBENCH_DIR)}  ({size_kb} KB)")

In [ ]:
import json, pathlib

# ── Find the R-Bench annotation files ─────────────────────────────────────
# The key files are: image-level_filterd.json and instance-level_filterd.json
# (note: "filterd" is the original authors' spelling, not a typo)

# Search for them in the entire R-Bench directory tree
all_jsons = sorted(pathlib.Path(RBENCH_DIR).rglob("*.json"))
print(f"All JSON files found ({len(all_jsons)}):")
for f in all_jsons:
    size_kb = f.stat().st_size // 1024
    print(f"  {f.relative_to(RBENCH_DIR)}  ({size_kb} KB)")

# ── Load the image-level questions (these are the Yes/No relational Q&A pairs)
rbench_data = None
ANNOTATION_PATH = None

# Priority: image-level_filterd.json (the main relational hallucination benchmark)
for f in all_jsons:
    fname = f.name.lower()
    if "image-level" in fname or "image_level" in fname:
        try:
            with open(f) as fh:
                data = json.load(fh)
            if isinstance(data, list) and len(data) > 50:
                rbench_data = data
                ANNOTATION_PATH = str(f)
                print(f"\n✅ Loaded image-level annotations: {f.name}")
                print(f"   Total entries: {len(data):,}")
                print(f"   Sample entry: {data[0]}")
                break
        except:
            pass

# Fallback: try any JSON that looks like a list of Q&A entries
if rbench_data is None:
    for f in all_jsons:
        try:
            with open(f) as fh:
                data = json.load(fh)
            if isinstance(data, list) and len(data) > 100:
                # Check if entries have question-like keys
                sample = str(data[0]).lower()
                if any(k in sample for k in ['question', 'text', 'label', 'answer']):
                    rbench_data = data
                    ANNOTATION_PATH = str(f)
                    print(f"\n✅ Loaded annotations from: {f.name}")
                    print(f"   Total entries: {len(data):,}")
                    print(f"   Sample entry keys: {list(data[0].keys()) if isinstance(data[0], dict) else 'N/A'}")
                    break
        except:
            pass

if rbench_data is None:
    print("\n⚠️  Could not find annotation files!")
    print("The Google Drive download may have failed or the file structure is different.")
    print("Try manually downloading from:")
    print("  https://drive.google.com/file/d/1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH/view")
    print(f"And extract to: {RBENCH_DIR}/data/")
    raise FileNotFoundError("R-Bench annotations not found")
else:
    print(f"\n   Full path: {ANNOTATION_PATH}")

In [ ]:
# ── Normalize R-Bench entries to a consistent schema ──────────────────────
# R-Bench JSON entries have keys like: image, question_id, text, label, type
# The 'image' field is a FILENAME (not a URL) — e.g. "val_00001234.jpg"
# Images come from Open Images V4 validation set (used by nocaps)

# First, let's see exactly what we're working with
print("RAW ENTRY KEYS:", list(rbench_data[0].keys()) if isinstance(rbench_data[0], dict) else "not a dict")
print("SAMPLE ENTRIES (first 3):")
for i, entry in enumerate(rbench_data[:3]):
    print(f"  [{i}] {entry}")
print()

def normalize_entry(entry: dict) -> dict:
    """Normalize a R-Bench entry to our schema."""
    def get(keys, default=None):
        for k in keys:
            if k in entry: return entry[k]
        return default

    # The 'image' field is a filename — store it so we can construct a download URL
    image_filename = get(["image", "img", "image_file"], "")

    # Extract image_id from filename (strip extension and path)
    if image_filename:
        img_id = os.path.splitext(os.path.basename(image_filename))[0]
    else:
        img_id = get(["image_id", "img_id", "id", "question_id"], str(hash(str(entry))))

    return {
        "image_id":       str(img_id),
        "image_filename": image_filename,       # raw filename from JSON
        "question_id":    get(["question_id", "qid"], ""),
        "question":       get(["text", "question", "q"], ""),
        "answer":         str(get(["label", "answer", "gt"], "")).strip().lower(),
        "relation_type":  get(["type", "relation_type", "rel_type", "qtype"], "unknown"),
    }

rbench_norm = [normalize_entry(e) for e in rbench_data]
rbench_norm = [e for e in rbench_norm if e['question'] and e['answer'] in ('yes','no')]

# Show distribution
import collections
rel_counts = collections.Counter(e['relation_type'] for e in rbench_norm)
print(f"Total usable questions: {len(rbench_norm):,}")
print("By relation type:")
for rel, cnt in rel_counts.most_common():
    print(f"  {rel}: {cnt:,}")

# Show unique image filenames
unique_images = set(e['image_filename'] for e in rbench_norm if e['image_filename'])
print(f"\nUnique images referenced: {len(unique_images):,}")
sample_fnames = sorted(unique_images)[:5]
print(f"Sample filenames: {sample_fnames}")
print(f"\nSample normalized entry: {rbench_norm[0]}")

In [ ]:
# ── Select 600 diverse images (balanced across relation types) ─────────────
# 600 images gives strong statistical power for WACV; R-Bench has 11,651 entries.
import random
random.seed(42)
SUBSET_SIZE = 600

from collections import defaultdict
by_image = defaultdict(list)
for entry in rbench_norm:
    by_image[entry['image_id']].append(entry)

all_image_ids = list(by_image.keys())
print(f"Total unique images in R-Bench: {len(all_image_ids):,}")

def has_variety(qlist):
    types = {q['relation_type'] for q in qlist}
    return len(types) >= 2

varied   = [iid for iid in all_image_ids if has_variety(by_image[iid])]
fallback = [iid for iid in all_image_ids if not has_variety(by_image[iid])]

selected_ids = (random.sample(varied,   min(SUBSET_SIZE, len(varied))) +
                random.sample(fallback, max(0, SUBSET_SIZE - len(varied))))[:SUBSET_SIZE]

id_to_filename = {}
for entry in rbench_norm:
    if entry['image_filename']:
        id_to_filename[entry['image_id']] = entry['image_filename']

print(f"Selected {len(selected_ids)} images ({len(varied)} with relation-type variety)")
print(f"Images with filenames: {sum(1 for s in selected_ids if s in id_to_filename)}/{len(selected_ids)}")

In [ ]:
# ── Download 200 nocaps images ─────────────────────────────────────────────
# nocaps images come from the Open Images V4 validation set.
# They're publicly hosted on AWS S3.
import requests
from tqdm.auto import tqdm
from pathlib import Path

EVAL_IMAGES_DIR = DRIVE_IMAGES_DIR  # ← persists in Google Drive across sessions
os.makedirs(EVAL_IMAGES_DIR, exist_ok=True)

# ── URL construction strategies ───────────────────────────────────────────
# Open Images V4 validation images are on S3 with the original filename.
# nocaps typically uses filenames like "0a1b2c3d4e5f6g7h.jpg" (Open Images IDs)
# or sometimes with subdirectory paths.

def get_download_urls(filename: str) -> list:
    """Generate candidate download URLs for a nocaps/Open Images image."""
    basename = os.path.basename(filename)
    urls = [
        # Open Images V4 validation set on S3 (most common)
        f"https://s3.amazonaws.com/open-images-dataset/validation/{basename}",
        # Alternative S3 endpoint
        f"https://open-images-dataset.s3.amazonaws.com/validation/{basename}",
        # nocaps direct (if hosted separately)
        f"https://nocaps.s3.amazonaws.com/val/{basename}",
    ]
    return urls

downloaded, skipped, failed = 0, 0, 0
image_id_to_path = {}
failed_ids = []

for iid in tqdm(selected_ids, desc="Downloading images"):
    filename = id_to_filename.get(iid, "")
    if not filename:
        failed += 1
        failed_ids.append((iid, "no filename"))
        continue

    # Use image_id (stem of filename) for local storage
    ext = os.path.splitext(filename)[1] or ".jpg"
    out_path = Path(f"{EVAL_IMAGES_DIR}/{iid}{ext}")

    if out_path.exists() and out_path.stat().st_size > 1000:
        skipped += 1
        image_id_to_path[iid] = str(out_path)
        continue

    # Try each URL pattern
    success = False
    for url in get_download_urls(filename):
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200 and len(resp.content) > 1000:
                out_path.write_bytes(resp.content)
                image_id_to_path[iid] = str(out_path)
                downloaded += 1
                success = True
                break
        except Exception:
            continue

    if not success:
        failed += 1
        failed_ids.append((iid, filename))

print(f"\n✅ Downloaded: {downloaded} | Already cached: {skipped} | Failed: {failed}")
print(f"Images available for eval: {len(image_id_to_path)}")

if failed_ids[:5]:
    print(f"\nFirst 5 failures (for debugging):")
    for fid, fname in failed_ids[:5]:
        print(f"  {fid} → {fname}")
    print(f"  Tried URLs like: {get_download_urls(failed_ids[0][1]) if failed_ids[0][1] != 'no filename' else 'N/A'}")

# ── If all downloads failed, the URL pattern is wrong ─────────────────────
# Fall back: try downloading images from R-Bench's own Google Drive images folder
if len(image_id_to_path) == 0:
    print("\n⚠️  All S3 downloads failed. The filename format may not match Open Images IDs.")
    print("Falling back to Google Drive images download...")
    print("Downloading R-Bench images from Google Drive (~1.6 GB)...")
    GDRIVE_IMAGES_ID = "10JXRdzTRMylWQA160qdoYITGO10g6N8k"
    IMAGES_ZIP = "/content/rbench_images_all.zip"
    !gdown --id {GDRIVE_IMAGES_ID} -O {IMAGES_ZIP}

    import zipfile, tarfile
    if os.path.exists(IMAGES_ZIP):
        EXTRACTED_DIR = "/content/rbench_images_extracted"
        os.makedirs(EXTRACTED_DIR, exist_ok=True)
        try:
            with zipfile.ZipFile(IMAGES_ZIP, 'r') as z:
                z.extractall(EXTRACTED_DIR)
            print(f"✅ Extracted to {EXTRACTED_DIR}")
        except zipfile.BadZipFile:
            try:
                with tarfile.open(IMAGES_ZIP, 'r:*') as t:
                    t.extractall(EXTRACTED_DIR)
                print(f"✅ Extracted tar to {EXTRACTED_DIR}")
            except:
                print("⚠️  Could not extract. Try manual download.")

        # Find images in extracted directory
        all_imgs = list(Path(EXTRACTED_DIR).rglob("*.jpg")) + list(Path(EXTRACTED_DIR).rglob("*.png"))
        print(f"Found {len(all_imgs)} images in extracted folder")

        # Build name → path index
        name_to_path = {img.stem: str(img) for img in all_imgs}
        name_to_path.update({img.name: str(img) for img in all_imgs})

        for iid in selected_ids:
            fname = id_to_filename.get(iid, "")
            basename = os.path.basename(fname)
            stem = os.path.splitext(basename)[0]
            # Try matching by stem or full name
            match = name_to_path.get(stem) or name_to_path.get(basename) or name_to_path.get(iid)
            if match:
                image_id_to_path[iid] = match

        print(f"Matched {len(image_id_to_path)} of {len(selected_ids)} selected images")

# ── Build eval dataset ────────────────────────────────────────────────────
eval_dataset = [
    {
        "image_id":   iid,
        "image_path": image_id_to_path[iid],
        "questions":  by_image[iid],
    }
    for iid in selected_ids
    if iid in image_id_to_path
]
print(f"\nEval dataset: {len(eval_dataset)} images with "
      f"{sum(len(d['questions']) for d in eval_dataset):,} questions")

# Save the subset info for reproducibility
import json
subset_path = f"{REPO_DIR}/eval/rbench_subset.json"
with open(subset_path, 'w') as f:
    json.dump(eval_dataset, f, indent=2)
print(f"Subset saved → {subset_path}")
import shutil as _sh; _sh.copy2(subset_path, DRIVE_EVAL_DIR); print(f"💾 Synced rbench_subset.json → Drive")

---
## Section 4: Baseline 1 — No Correction
Raw BLIP-2 answers to R-Bench Yes/No questions, no correction applied.
This is the lower bound for R-POPE.

In [ ]:
# ── Baseline 1: No Correction ─────────────────────────────────────────────
# BLIP-2 generates caption (not used for answering).
# LLaVA answers R-Bench questions directly from the image — no caption context.
# This is the lower bound: how well can LLaVA answer relational questions cold?
from tqdm.auto import tqdm
import pandas as pd, time

print("Running Baseline 1: No Correction (LLaVA direct VQA) ...")
baseline1_rows, baseline1_preds = [], []
b1_captions = {}   # store BLIP-2 captions for downstream use

for item in tqdm(eval_dataset, desc="Baseline 1"):
    image = Image.open(item['image_path']).convert("RGB")
    # Generate BLIP-2 caption (stored for self-refine and RelCheck)
    caption = generate_caption(image)
    b1_captions[item['image_id']] = caption

    for q in item['questions']:
        pred = llava_direct(image, q['question'])
        gt   = q['answer'].strip().lower()
        baseline1_preds.append({'pred': pred, 'gt': gt})
        baseline1_rows.append({
            'image_id':      item['image_id'],
            'question':      q['question'],
            'gt':            gt,
            'pred':          pred,
            'correct':       (pred == gt),
            'relation_type': q['relation_type'],
            'blip2_caption': caption,
        })

df_b1 = pd.DataFrame(baseline1_rows)
B1_CSV = f"{REPO_DIR}/eval/baseline_no_correction.csv"
df_b1.to_csv(B1_CSV, index=False)
import shutil as _shutil; _shutil.copy2(B1_CSV, DRIVE_EVAL_DIR); print(f"💾 Synced baseline_no_correction.csv → Drive")

b1_metrics = compute_rpope_metrics(baseline1_preds)
print(f"\n{'='*55}")
print("BASELINE 1 (No Correction) — R-POPE  [LLaVA evaluator]")
print(f"{'='*55}")
for k, v in b1_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"Saved → {B1_CSV}")

---
## Section 5: Baseline 2 — Self-Refinement
Ask BLIP-2 to re-examine its own description, then re-run VQA on the refined caption.

In [ ]:
# ── Baseline 2: Self-Refinement ───────────────────────────────────────────
# BLIP-2 generates caption, then is asked to self-correct it.
# LLaVA answers R-Bench questions WITH the self-refined caption as context.
# Shows that naive self-refinement (same model) doesn't reliably fix hallucinations.
print("Running Baseline 2: Self-Refinement (LLaVA evaluator + BLIP-2 context) ...")
baseline2_rows, baseline2_preds = [], []

def self_refine_caption(image: Image.Image, original_caption: str) -> str:
    """Ask BLIP-2 to self-correct its own caption."""
    refine_prompt = (
        f"Initial description: {original_caption}\n"
        "Look at the image carefully. Is the above description accurate? "
        "If any relationships between objects are wrong, correct only those parts."
    )
    inputs = blip2_processor(
        images=image, text=refine_prompt, return_tensors="pt"
    ).to(blip2_model.device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=100)
    refined = blip2_processor.decode(ids[0], skip_special_tokens=True).strip()
    if len(refined) < 20 or refined.lower().startswith("yes"):
        return original_caption
    return refined

for item in tqdm(eval_dataset, desc="Baseline 2"):
    image           = Image.open(item['image_path']).convert("RGB")
    original_cap    = b1_captions.get(item['image_id']) or generate_caption(image)
    refined_caption = self_refine_caption(image, original_cap)

    for q in item['questions']:
        pred = llava_with_caption(image, refined_caption, q['question'])
        gt   = q['answer'].strip().lower()
        baseline2_preds.append({'pred': pred, 'gt': gt})
        baseline2_rows.append({
            'image_id':        item['image_id'],
            'question':        q['question'],
            'original_caption': original_cap,
            'refined_caption': refined_caption,
            'gt':              gt,
            'pred':            pred,
            'correct':         (pred == gt),
            'relation_type':   q['relation_type'],
        })

df_b2 = pd.DataFrame(baseline2_rows)
B2_CSV = f"{REPO_DIR}/eval/baseline_self_refine.csv"
df_b2.to_csv(B2_CSV, index=False)
import shutil as _shutil; _shutil.copy2(B2_CSV, DRIVE_EVAL_DIR); print(f"💾 Synced baseline_self_refine.csv → Drive")

b2_metrics = compute_rpope_metrics(baseline2_preds)
print(f"\n{'='*55}")
print("BASELINE 2 (Self-Refinement) — R-POPE  [LLaVA evaluator]")
print(f"{'='*55}")
for k, v in b2_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"Saved → {B2_CSV}")

---
## Section 6: Full RelCheck
Runs the complete 3-stage pipeline on all 200 images.
For each R-Bench question, uses verified triples where available, falls back to VQA otherwise.

In [ ]:
# ── Full RelCheck ─────────────────────────────────────────────────────────
# BLIP-2 generates captions → RelCheck corrects them (LLaVA verifies, Mistral corrects)
# LLaVA answers R-Bench questions WITH RelCheck's corrected caption as context.
# Directly measures: do better (corrected) captions improve downstream relational VQA?
import Levenshtein
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

print("Running Full RelCheck on 600 images ...")
relcheck_rows, relcheck_preds = [], []
pipeline_results_cache = {}
smoother = SmoothingFunction().method1

for item in tqdm(eval_dataset, desc="RelCheck"):
    image       = Image.open(item['image_path']).convert("RGB")
    orig_caption = b1_captions.get(item['image_id']) or generate_caption(image)

    try:
        result = pipeline.run(
            image_path      = item['image_path'],
            caption         = orig_caption,       # reuse BLIP-2 caption from Baseline 1
            blip2_processor = blip2_processor,
            blip2_model     = blip2_model,
        )
        pipeline_results_cache[item['image_id']] = result
    except Exception as e:
        print(f"  Pipeline error on {item['image_id']}: {e}")
        # Use original caption so prediction count stays aligned with baselines
        class _FallbackResult: pass
        result = _FallbackResult()
        result.original_caption = orig_caption
        result.corrected_caption = orig_caption
        result.n_triples = 0
        result.n_hallucinated = 0
        result.edit_rate = 0.0

    ref_tokens = result.original_caption.split()
    hyp_tokens = result.corrected_caption.split()
    bleu4 = sentence_bleu(
        [ref_tokens], hyp_tokens,
        weights=(0.25,)*4, smoothing_function=smoother
    )

    for q in item['questions']:
        # LLaVA evaluates with corrected caption as context
        pred = llava_with_caption(image, result.corrected_caption, q['question'])
        gt   = q['answer'].strip().lower()
        relcheck_preds.append({'pred': pred, 'gt': gt})
        relcheck_rows.append({
            'image_id':          item['image_id'],
            'question':          q['question'],
            'gt':                gt,
            'pred':              pred,
            'correct':           (pred == gt),
            'relation_type':     q['relation_type'],
            'original_caption':  result.original_caption,
            'corrected_caption': result.corrected_caption,
            'n_triples':         result.n_triples,
            'n_hallucinated':    result.n_hallucinated,
            'edit_rate':         result.edit_rate,
            'bleu4':             bleu4,
        })

df_rc = pd.DataFrame(relcheck_rows)
RC_CSV = f"{REPO_DIR}/eval/relcheck_results.csv"
df_rc.to_csv(RC_CSV, index=False)
import shutil as _shutil; _shutil.copy2(RC_CSV, DRIVE_EVAL_DIR); print(f"💾 Synced relcheck_results.csv → Drive")

rc_metrics = compute_rpope_metrics(relcheck_preds)
print(f"\n{'='*55}")
print("FULL RELCHECK — R-POPE  [LLaVA evaluator]")
print(f"{'='*55}")
for k, v in rc_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")

df_rc_img = df_rc.groupby('image_id').first()
print(f"\n  Avg edit_rate : {df_rc_img['edit_rate'].mean():.4f}")
print(f"  Avg BLEU-4    : {df_rc_img['bleu4'].mean():.4f}")
print(f"  Images corrected: {(df_rc_img['edit_rate'] > 0).sum()}/{len(df_rc_img)}")
print(f"Saved → {RC_CSV}")

In [ ]:
# ── Sanity check: did RelCheck actually detect and correct anything? ────────
import pandas as pd
df_rc_check = pd.read_csv(RC_CSV)

# Per-image stats (deduplicated)
df_img = df_rc_check.groupby('image_id').first().reset_index()

n_with_triples    = (df_img['n_triples']     > 0).sum()
n_with_hallu      = (df_img['n_hallucinated'] > 0).sum()
n_with_correction = (df_img['edit_rate']      > 0).sum()

print("=" * 55)
print("RELCHECK SANITY CHECK")
print("=" * 55)
print(f"  Images with ≥1 triple extracted : {n_with_triples}/{len(df_img)}")
print(f"  Images with ≥1 hallucination    : {n_with_hallu}/{len(df_img)}")
print(f"  Images with edit_rate > 0       : {n_with_correction}/{len(df_img)}")
print(f"  Avg edit_rate                   : {df_img['edit_rate'].mean():.4f}")
print(f"  Avg BLEU-4                      : {df_img['bleu4'].mean():.4f}")

print("\nDIAGNOSIS:")
if n_with_triples == 0:
    print("  ❌ Stage 1 broken — no triples extracted. Check spaCy.")
elif n_with_hallu == 0:
    print("  ❌ Stage 2 broken — verifier never flags hallucinations.")
    print("     Pull latest relation_verifier.py (token-prob fix) and rerun.")
elif n_with_correction == 0:
    print("  ❌ Stage 3 broken — corrector not firing despite hallucinations found.")
else:
    print(f"  ✅ Pipeline working end-to-end.")
    print(f"     {n_with_hallu} images had hallucinations; {n_with_correction} were corrected.")

# Show a few examples
print("\nSample rows (n_triples, n_hallucinated, edit_rate):")
print(df_img[['image_id','n_triples','n_hallucinated','edit_rate']].head(10).to_string(index=False))

---
## Section 7: Ablation Study
Three ablated variants to show each component contributes independently:
1. **No Spatial Verifier** — OWL-ViT disabled, LLaVA handles all spatial relations
2. **No LLaVA Verifier** — cross-model VQA disabled (falls back to BLIP-2 VQA, showing the circular-verification degradation)
3. **Detect-Only** — stages 1+2 run, stage 3 (correction) skipped
4. **spaCy Extractor** — LLM extractor replaced with spaCy (shows LLM extraction quality)

In [ ]:
ablation_configs = [
    dict(name="no_spatial",   skip_spatial=True,  skip_vqa=False, detection_only=False, use_llm=True),
    dict(name="no_llava_vqa", skip_spatial=False, skip_vqa=False, detection_only=False, use_llm=True,  no_llava=True),
    dict(name="detect_only",  skip_spatial=False, skip_vqa=False, detection_only=True,  use_llm=True),
    dict(name="spacy_extractor", skip_spatial=False, skip_vqa=False, detection_only=False, use_llm=False),
]

ablation_summary = []

for config in ablation_configs:
    name     = config.pop("name")
    use_llm  = config.pop("use_llm")
    no_llava = config.pop("no_llava", False)

    print(f"\n{'='*50}\nAblation: {name}\n{'='*50}")

    abl_pipeline = RelCheckPipeline(
        together_api_key  = TOGETHER_API_KEY,
        llava_model       = None if no_llava else llava_model,
        llava_processor   = None if no_llava else llava_processor,
        use_llm_extractor = use_llm,
        **config,
    )

    abl_rows, abl_preds = [], []

    for item in tqdm(eval_dataset, desc=name):
        image       = Image.open(item['image_path']).convert("RGB")
        orig_caption = b1_captions.get(item['image_id']) or generate_caption(image)
        try:
            result = abl_pipeline.run(
                image_path      = item['image_path'],
                caption         = orig_caption,
                blip2_processor = blip2_processor,
                blip2_model     = blip2_model,
            )
        except Exception as e:
            print(f"  Error on {item['image_id']}: {e}")
            continue

        for q in item['questions']:
            pred = llava_with_caption(image, result.corrected_caption, q['question'])
            gt   = q['answer'].strip().lower()
            abl_preds.append({'pred': pred, 'gt': gt})
            abl_rows.append({
                'ablation': name, 'image_id': item['image_id'],
                'question': q['question'], 'gt': gt, 'pred': pred,
                'correct': (pred == gt), 'relation_type': q['relation_type'],
                'n_hallucinated': result.n_hallucinated, 'edit_rate': result.edit_rate,
            })

    metrics = compute_rpope_metrics(abl_preds)
    print(f"  Accuracy={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}")
    ablation_summary.append({'variant': name, **metrics})

    df_abl = pd.DataFrame(abl_rows)
    csv_path = f"{REPO_DIR}/eval/ablation_{name}.csv"
    df_abl.to_csv(csv_path, index=False)
    print(f"  Saved → {csv_path}")

# Add full RelCheck as the reference row
ablation_summary.append({'variant': 'RelCheck (full)', **rc_metrics})

df_ablation = pd.DataFrame(ablation_summary)
ABL_CSV = f"{REPO_DIR}/eval/ablation_results.csv"
df_ablation.to_csv(ABL_CSV, index=False)
print(f"\n✅ Ablation summary saved → {ABL_CSV}")
import shutil as _shutil; _shutil.copy2(ABL_CSV, DRIVE_EVAL_DIR); print(f"💾 Synced ablation_results.csv → Drive")
display(df_ablation[['variant','accuracy','precision','recall','f1']].round(4))

---
## Section 8: R-CHAIR Annotation (50 images)
Manually annotate whether each extracted triple is a hallucination.
Computes R-CHAIR_s (caption-level) and R-CHAIR_i (triple-level) metrics.

**This section requires 30–60 minutes of manual annotation.**
Each cell presents one image + its extracted triples; you enter 0 (correct) or 1 (hallucinated) for each.

---
## Section 8: Woodpecker Baseline
Woodpecker (Yin et al., 2023) is the closest prior work — it corrects *object* hallucinations using a similar detect-then-correct approach but ignores relational structure entirely.

We implement Woodpecker's core evaluation loop: VQA-based object presence checking, no relation-type routing, no spatial geometry verification. This shows RelCheck's relational specialization provides measurable gains on relational benchmarks.

In [ ]:
# ── Woodpecker Baseline ───────────────────────────────────────────────────
# Woodpecker (Yin et al. 2023) corrects object hallucinations via:
#   1. Extract key concepts (nouns) from caption
#   2. VQA-verify each object is present in the image
#   3. Remove/replace hallucinated objects with LLM correction
#
# Critically: Woodpecker has NO relation-type routing, NO spatial geometry check,
# and does NOT verify subject-relation-object triples — only object existence.
# This is the key architectural gap RelCheck addresses.

from triple_extractor import TripleExtractor as _SpacyExtractor
import together as _together

_spacy_ext    = _SpacyExtractor()
_tog_client   = _together.Together(api_key=TOGETHER_API_KEY)

def woodpecker_verify_object(image: Image.Image, obj: str) -> bool:
    """Check if an object is present in the image (Woodpecker-style: VQA only)."""
    question = f"Is there a {obj} in this image? Answer yes or no only."
    return llava_direct(image, question) == "yes"

def woodpecker_correct(caption: str, hallucinated_objects: list[str]) -> str:
    """Use Llama-3.3-70B to remove hallucinated objects from caption (Woodpecker-style)."""
    if not hallucinated_objects:
        return caption
    obj_list = ", ".join(f'"{o}"' for o in hallucinated_objects)
    prompt = (
        f"Caption: \"{caption}\"\n"
        f"The following objects are NOT present in the image: {obj_list}.\n"
        "Rewrite the caption removing or replacing references to these objects. "
        "Keep all other content unchanged. Output only the corrected caption."
    )
    try:
        resp = _tog_client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=120, temperature=0.0,
        )
        return resp.choices[0].message.content.strip().strip('"')
    except:
        return caption

def run_woodpecker(image: Image.Image, caption: str) -> str:
    """Woodpecker pipeline: extract nouns → verify presence → correct."""
    triples = _spacy_ext.extract(caption)
    # Collect unique noun objects/subjects from triples
    nouns = set()
    for t in triples:
        nouns.add(t.subject.split()[-1])   # head noun of subject phrase
        nouns.add(t.obj.split()[-1])       # head noun of object phrase
    nouns = {n for n in nouns if len(n) > 2}

    hallucinated_objs = [n for n in nouns if not woodpecker_verify_object(image, n)]
    return woodpecker_correct(caption, hallucinated_objs)

print("Running Woodpecker baseline on 600 images ...")
woodpecker_rows, woodpecker_preds = [], []

for item in tqdm(eval_dataset, desc="Woodpecker"):
    image        = Image.open(item['image_path']).convert("RGB")
    orig_caption = b1_captions.get(item['image_id']) or generate_caption(image)
    wp_caption   = run_woodpecker(image, orig_caption)

    for q in item['questions']:
        pred = llava_with_caption(image, wp_caption, q['question'])
        gt   = q['answer'].strip().lower()
        woodpecker_preds.append({'pred': pred, 'gt': gt})
        woodpecker_rows.append({
            'image_id':       item['image_id'],
            'question':       q['question'],
            'gt':             gt, 'pred': pred,
            'correct':        (pred == gt),
            'relation_type':  q['relation_type'],
            'orig_caption':   orig_caption,
            'wp_caption':     wp_caption,
        })

df_wp = pd.DataFrame(woodpecker_rows)
WP_CSV = f"{REPO_DIR}/eval/baseline_woodpecker.csv"
df_wp.to_csv(WP_CSV, index=False)
import shutil as _shutil; _shutil.copy2(WP_CSV, DRIVE_EVAL_DIR); print(f"💾 Synced baseline_woodpecker.csv → Drive")

wp_metrics = compute_rpope_metrics(woodpecker_preds)
print(f"\n{'='*55}")
print("WOODPECKER BASELINE — R-POPE  [LLaVA evaluator]")
print(f"{'='*55}")
for k, v in wp_metrics.items():
    if isinstance(v, float): print(f"  {k:12s}: {v:.4f}")
    else:                    print(f"  {k:12s}: {v}")
print(f"Saved → {WP_CSV}")

---
## Section 9: Generalizability — InstructBLIP as Second MLLM
RelCheck is designed to be model-agnostic: it operates post-hoc on any caption string.
Here we apply RelCheck to captions generated by **InstructBLIP-FlanT5-XL** (Dai et al., 2023) — a different architecture from BLIP-2 — and measure R-POPE improvement.

If RelCheck improves R-POPE for *both* BLIP-2 and InstructBLIP, it demonstrates the system generalizes beyond a single captioning model.

Note: InstructBLIP uses a different visual encoder (EVA-CLIP) and instruction-tuned language model, making it architecturally distinct from BLIP-2.

In [ ]:
# ── Load InstructBLIP-FlanT5-XL (second MLLM for generalizability) ────────
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

IBLIP_MODEL_ID = "Salesforce/instructblip-flan-t5-xl"
print(f"Loading {IBLIP_MODEL_ID} in 8-bit ...")

iblip_processor = InstructBlipProcessor.from_pretrained(IBLIP_MODEL_ID)
iblip_model = InstructBlipForConditionalGeneration.from_pretrained(
    IBLIP_MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
iblip_model.eval()
torch.cuda.empty_cache()
print(f"✅ InstructBLIP loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── InstructBLIP: generate captions + run RelCheck + evaluate ─────────────
def generate_iblip_caption(image: Image.Image, max_tokens: int = 80) -> str:
    """Generate an InstructBLIP caption."""
    inputs = iblip_processor(
        images=image,
        text="Describe this image in detail.",
        return_tensors="pt"
    ).to(iblip_model.device)
    with torch.no_grad():
        ids = iblip_model.generate(**inputs, max_new_tokens=max_tokens)
    return iblip_processor.decode(ids[0], skip_special_tokens=True).strip()

# Baseline: InstructBLIP captions, LLaVA evaluates directly (no context)
print("InstructBLIP — Baseline (no correction) ...")
iblip_baseline_preds, iblip_rc_preds = [], []
iblip_captions = {}

for item in tqdm(eval_dataset[:200], desc="InstructBLIP baseline"):  # 200-image subset for speed
    image   = Image.open(item['image_path']).convert("RGB")
    caption = generate_iblip_caption(image)
    iblip_captions[item['image_id']] = caption
    for q in item['questions']:
        pred = llava_with_caption(image, caption, q['question'])
        iblip_baseline_preds.append({'pred': pred, 'gt': q['answer']})

# RelCheck on InstructBLIP captions
print("InstructBLIP — RelCheck ...")
iblip_pipeline = RelCheckPipeline(
    together_api_key  = TOGETHER_API_KEY,
    llava_model       = llava_model,
    llava_processor   = llava_processor,
    use_llm_extractor = True,
)

for item in tqdm(eval_dataset[:200], desc="InstructBLIP + RelCheck"):
    image        = Image.open(item['image_path']).convert("RGB")
    iblip_cap    = iblip_captions.get(item['image_id']) or generate_iblip_caption(image)
    try:
        result = iblip_pipeline.run(
            image_path=item['image_path'], caption=iblip_cap,
            blip2_processor=iblip_processor, blip2_model=iblip_model,
        )
    except Exception as e:
        result_caption = iblip_cap
    else:
        result_caption = result.corrected_caption

    for q in item['questions']:
        pred = llava_with_caption(image, result_caption, q['question'])
        iblip_rc_preds.append({'pred': pred, 'gt': q['answer']})

iblip_b_metrics  = compute_rpope_metrics(iblip_baseline_preds)
iblip_rc_metrics = compute_rpope_metrics(iblip_rc_preds)

print(f"\n{'='*55}")
print("GENERALIZABILITY: InstructBLIP  [LLaVA evaluator, 200 images]")
print(f"{'='*55}")
print(f"  {'Method':<30} {'Acc':>7} {'F1':>7}")
print(f"  {'InstructBLIP (no correction)':<30} {iblip_b_metrics['accuracy']:>7.4f} {iblip_b_metrics['f1']:>7.4f}")
print(f"  {'InstructBLIP + RelCheck':<30} {iblip_rc_metrics['accuracy']:>7.4f} {iblip_rc_metrics['f1']:>7.4f}")
delta_acc = iblip_rc_metrics['accuracy'] - iblip_b_metrics['accuracy']
delta_f1  = iblip_rc_metrics['f1']       - iblip_b_metrics['f1']
print(f"  {'Δ (RelCheck improvement)':<30} {delta_acc:>+7.4f} {delta_f1:>+7.4f}")

import json
gen_path = f"{REPO_DIR}/eval/generalizability_instructblip.json"
with open(gen_path, 'w') as f:
    json.dump({'baseline': iblip_b_metrics, 'relcheck': iblip_rc_metrics}, f, indent=2)
print(f"Saved → {gen_path}")

---
## Section 10: Statistical Significance
Bootstrap confidence intervals (n=10,000 resamples) for all R-POPE metrics.
McNemar's test for pairwise comparison of RelCheck vs each baseline.

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

np.random.seed(42)
N_BOOTSTRAP = 10_000

def bootstrap_ci(preds: list[dict], metric: str = 'accuracy', alpha: float = 0.05):
    """Bootstrap confidence interval for a R-POPE metric."""
    n = len(preds)
    scores = []
    for _ in range(N_BOOTSTRAP):
        sample = [preds[i] for i in np.random.randint(0, n, n)]
        scores.append(compute_rpope_metrics(sample)[metric])
    lower = np.percentile(scores, 100 * alpha / 2)
    upper = np.percentile(scores, 100 * (1 - alpha / 2))
    return lower, upper

def mcnemar_test(preds_a: list[dict], preds_b: list[dict]):
    """
    McNemar's test: are two classifiers significantly different?
    Returns p-value. Both lists must be aligned (same questions, same order).
    """
    assert len(preds_a) == len(preds_b), "Prediction lists must be aligned"
    # Count discordant pairs
    b = sum(1 for a, b_ in zip(preds_a, preds_b)
            if (a['pred'] == a['gt']) and (b_['pred'] != b_['gt']))
    c = sum(1 for a, b_ in zip(preds_a, preds_b)
            if (a['pred'] != a['gt']) and (b_['pred'] == b_['gt']))
    # McNemar with continuity correction
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    from scipy.stats import chi2 as chi2_dist
    return 1 - chi2_dist.cdf(chi2, df=1)

print("Computing bootstrap confidence intervals ...")
print(f"{'Method':<30} {'Accuracy':>10} {'95% CI':>20} {'F1':>8}")
print("-" * 72)

all_preds = {
    'No Correction':  baseline1_preds,
    'Self-Refine':    baseline2_preds,
    'Woodpecker':     woodpecker_preds,
    'RelCheck':       relcheck_preds,
}
ci_results = {}
for name, preds in all_preds.items():
    acc = compute_rpope_metrics(preds)['accuracy']
    f1  = compute_rpope_metrics(preds)['f1']
    lo, hi = bootstrap_ci(preds, 'accuracy')
    ci_results[name] = {'accuracy': acc, 'ci_lo': lo, 'ci_hi': hi, 'f1': f1}
    print(f"  {name:<28} {acc:>10.4f} [{lo:.4f}, {hi:.4f}]  {f1:>8.4f}")

print("\nMcNemar's test (RelCheck vs each baseline):")
for name, preds in all_preds.items():
    if name == 'RelCheck':
        continue
    n = min(len(relcheck_preds), len(preds))
    p = mcnemar_test(relcheck_preds[:n], preds[:n])
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"  RelCheck vs {name:<20} p={p:.4f}  {sig}")

# Save
import json
stat_path = f"{REPO_DIR}/eval/statistical_significance.json"
with open(stat_path, 'w') as f:
    json.dump(ci_results, f, indent=2)
print(f"\nSaved → {stat_path}")

In [ ]:
from IPython.display import display as ipy_display, Image as IPyImage
import ipywidgets as widgets

# Use first 50 images from eval set
RCHAIR_SUBSET = eval_dataset[:50]
rchair_labels = {}  # image_id → list of {triple_str, hallucinated}

# Pre-populate using pipeline results already computed
for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid in pipeline_results_cache:
        result = pipeline_results_cache[iid]
        rchair_labels[iid] = [
            {
                'triple': str(t),
                'relcheck_judgment': t.hallucinated,
                'human_judgment': None,  # to be filled below
            }
            for t in result.triples
        ]

print(f"R-CHAIR annotation ready for {len(RCHAIR_SUBSET)} images.")
print(f"{sum(len(v) for v in rchair_labels.values())} triples to annotate.")
print("\nRun the next cell to start the annotation interface.")

In [ ]:
# ── Interactive annotation loop ────────────────────────────────────────────
# For each image, shows the image + triples and asks you to confirm/override
# RelCheck's judgment.
#
# Press Enter to accept RelCheck's judgment, or type 'y'/'n' to override.

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

annotated_count = 0

for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid not in rchair_labels:
        continue

    triples_info = rchair_labels[iid]
    if not triples_info:
        continue

    # Show image
    fig, ax = plt.subplots(figsize=(5,5))
    ax.imshow(mpimg.imread(item['image_path']))
    ax.axis('off')
    ax.set_title(f"Image: {iid}", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nCaption: {pipeline_results_cache[iid].original_caption}")
    print("Triples (RelCheck judgment shown):")

    for i, t_info in enumerate(triples_info):
        rc_j = t_info['relcheck_judgment']
        rc_label = "HALLUCINATED" if rc_j else ("verified" if rc_j is False else "uncertain")
        resp = input(f"  [{i+1}] {t_info['triple']} — RelCheck says: {rc_label}. "
                     f"Hallucinated? (y/n/Enter=agree): ").strip().lower()
        if resp == 'y':
            t_info['human_judgment'] = True
        elif resp == 'n':
            t_info['human_judgment'] = False
        else:
            t_info['human_judgment'] = rc_j  # agree with RelCheck

    annotated_count += 1
    if annotated_count % 10 == 0:
        print(f"\n  [Progress: {annotated_count}/50 images annotated]\n")

print(f"\n✅ Annotation complete for {annotated_count} images.")

In [ ]:
# ── Compute R-CHAIR metrics ────────────────────────────────────────────────
rchair_rows = []
captions_with_hallu  = 0
total_captions       = 0
hallucinated_triples = 0
total_triples        = 0

for item in RCHAIR_SUBSET:
    iid = item['image_id']
    if iid not in rchair_labels:
        continue

    triples_info = rchair_labels[iid]
    judged = [t for t in triples_info if t['human_judgment'] is not None]
    if not judged:
        continue

    total_captions  += 1
    total_triples   += len(judged)
    n_hallu = sum(1 for t in judged if t['human_judgment'] is True)
    hallucinated_triples += n_hallu
    if n_hallu > 0:
        captions_with_hallu += 1

    for t_info in judged:
        rchair_rows.append({
            'image_id':        iid,
            'triple':          t_info['triple'],
            'relcheck_judgment': t_info['relcheck_judgment'],
            'human_judgment':  t_info['human_judgment'],
        })

rchair_s = captions_with_hallu / total_captions if total_captions else 0
rchair_i = hallucinated_triples / total_triples if total_triples else 0

df_rchair = pd.DataFrame(rchair_rows)
RCHAIR_CSV = f"{REPO_DIR}/eval/rchair_labels.csv"
df_rchair.to_csv(RCHAIR_CSV, index=False)

print(f"\n{'='*50}")
print("R-CHAIR METRICS")
print(f"{'='*50}")
print(f"  R-CHAIR_s (caption-level): {rchair_s:.4f}  "
      f"({captions_with_hallu}/{total_captions} captions have ≥1 hallucination)")
print(f"  R-CHAIR_i (triple-level):  {rchair_i:.4f}  "
      f"({hallucinated_triples}/{total_triples} triples hallucinated)")
print(f"\nLabels saved → {RCHAIR_CSV}")

---
## Section 9: Results Tables & Figures
Generates publication-ready figures saved to `figures/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figure 1: Main results bar chart (R-POPE accuracy + F1) ───────────────
conditions = ["No Correction", "Self-Refinement", "RelCheck (ours)"]
accuracy   = [b1_metrics['accuracy'], b2_metrics['accuracy'], rc_metrics['accuracy']]
f1         = [b1_metrics['f1'],       b2_metrics['f1'],       rc_metrics['f1']]
precision  = [b1_metrics['precision'],b2_metrics['precision'],rc_metrics['precision']]
recall     = [b1_metrics['recall'],   b2_metrics['recall'],   rc_metrics['recall']]

x = np.arange(len(conditions))
width = 0.2
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(9, 5))
bars_acc  = ax.bar(x - 1.5*width, accuracy,  width, label='Accuracy',  color=colors[0])
bars_prec = ax.bar(x - 0.5*width, precision, width, label='Precision', color=colors[1])
bars_rec  = ax.bar(x + 0.5*width, recall,    width, label='Recall',    color=colors[2])
bars_f1   = ax.bar(x + 1.5*width, f1,        width, label='F1',        color=colors[3])

ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('R-POPE: Relational Hallucination Detection\n(200-image R-Bench subset)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars_acc, bars_prec, bars_rec, bars_f1]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)

plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/rpope_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/rpope_results.png")

In [ ]:
# ── Figure 2: Ablation Study ───────────────────────────────────────────────
df_abl_plot = pd.read_csv(ABL_CSV)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

variants = df_abl_plot['variant'].tolist()
colors_abl = ['#8172B3', '#64B5CD', '#CCB974', '#4C72B0']

# Accuracy
axes[0].bar(variants, df_abl_plot['accuracy'], color=colors_abl)
axes[0].set_title('Ablation: R-POPE Accuracy', fontsize=12)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.0)
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_abl_plot['accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# F1
axes[1].bar(variants, df_abl_plot['f1'], color=colors_abl)
axes[1].set_title('Ablation: R-POPE F1', fontsize=12)
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0, 1.0)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_abl_plot['f1']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Ablation Study — RelCheck Module Contributions', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/ablation_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/ablation_results.png")

In [ ]:
# ── Figure 3: R-POPE Accuracy broken down by relation type ────────────────
df_rc_rt = pd.read_csv(RC_CSV)
df_b1_rt = pd.read_csv(B1_CSV)

rel_types = df_rc_rt['relation_type'].unique()
rc_by_rel = df_rc_rt.groupby('relation_type')['correct'].mean()
b1_by_rel = df_b1_rt.groupby('relation_type')['correct'].mean()

common_rels = sorted(set(rc_by_rel.index) & set(b1_by_rel.index))
x = np.arange(len(common_rels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, [b1_by_rel[r] for r in common_rels], w,
       label='No Correction', color='#4C72B0')
ax.bar(x + w/2, [rc_by_rel[r] for r in common_rels], w,
       label='RelCheck (ours)', color='#DD8452')

ax.set_xticks(x)
ax.set_xticklabels([r.capitalize() for r in common_rels], fontsize=11)
ax.set_ylabel('R-POPE Accuracy')
ax.set_title('R-POPE Accuracy by Relation Type', fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(f"{REPO_DIR}/figures/accuracy_by_relation.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved → figures/accuracy_by_relation.png")

In [ ]:
# ── Figure 4: Qualitative before/after examples ───────────────────────────
# Pick 5 images where RelCheck actually made a correction

corrected_examples = [
    (iid, res)
    for iid, res in pipeline_results_cache.items()
    if res.n_corrected > 0 and res.original_caption != res.corrected_caption
][:5]

if len(corrected_examples) < 5:
    print(f"Only {len(corrected_examples)} corrected examples found.")

n = len(corrected_examples)
if n == 0:
    print("No corrections found to display. Try running on more images.")
else:
    fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for ax, (iid, res) in zip(axes, corrected_examples):
        img_path = next(
            (d['image_path'] for d in eval_dataset if d['image_id'] == iid),
            None
        )
        if img_path and os.path.exists(img_path):
            ax.imshow(mpimg.imread(img_path))
        ax.axis('off')
        title = (
            f"ORIGINAL:  {res.original_caption}\n"
            f"CORRECTED: {res.corrected_caption}"
        )
        ax.set_title(title, fontsize=9, loc='left', pad=8, wrap=True)

    plt.suptitle('RelCheck: Qualitative Before/After Examples', fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(f"{REPO_DIR}/figures/qualitative_examples.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → figures/qualitative_examples.png")

In [ ]:
# ── Final Summary Table ────────────────────────────────────────────────────
print("\n" + "="*70)
print("RELCHECK — FINAL RESULTS SUMMARY")
print("="*70)

summary = pd.DataFrame([
    {"Method": "No Correction",    **{k: round(v,4) for k,v in b1_metrics.items() if isinstance(v,float)}},
    {"Method": "Self-Refinement",  **{k: round(v,4) for k,v in b2_metrics.items() if isinstance(v,float)}},
    {"Method": "RelCheck (ours)",  **{k: round(v,4) for k,v in rc_metrics.items() if isinstance(v,float)}},
])[['Method', 'accuracy', 'precision', 'recall', 'f1', 'yes_ratio']]

display(summary)

print(f"\nAdditional RelCheck metrics:")
df_rc_img = df_rc.groupby('image_id').first().reset_index()
print(f"  Avg Edit Rate:   {df_rc_img['edit_rate'].mean():.4f}")
print(f"  Avg BLEU-4:      {df_rc_img['bleu4'].mean():.4f}")
if total_captions > 0:
    print(f"  R-CHAIR_s:       {rchair_s:.4f}")
    print(f"  R-CHAIR_i:       {rchair_i:.4f}")

print(f"\nFigures saved to: {REPO_DIR}/figures/")
print(f"CSVs saved to:    {REPO_DIR}/eval/")
print("\n✅ All done! Push results to GitHub:")
print(f"  cd {REPO_DIR} && git add eval/ figures/ && "
      f"git commit -m 'Add evaluation results' && git push")

In [ ]:
# ── Sync results to Google Drive ─────────────────────────────────────────
# Copies all CSVs, JSONs, and figures to Drive so results survive
# even if you haven't pushed to GitHub yet.
import shutil, glob

REPO_DIR = "/content/RelCheck"  # defined earlier, re-stating for clarity

for pattern in [
    f"{REPO_DIR}/eval/*.csv",
    f"{REPO_DIR}/eval/*.json",
    f"{REPO_DIR}/figures/*.png",
    f"{REPO_DIR}/figures/*.pdf",
]:
    for f in glob.glob(pattern):
        dest_dir = DRIVE_EVAL_DIR if '/eval/' in f else DRIVE_FIGURES_DIR
        shutil.copy2(f, dest_dir)
        print(f"  Copied: {os.path.basename(f)} → {dest_dir}")

print(f"\n✅ All results synced to Google Drive")
print(f"   CSVs + JSONs → {DRIVE_EVAL_DIR}")
print(f"   Figures      → {DRIVE_FIGURES_DIR}")
print(f"\nTo access your files:")
print(f"  1. Go to drive.google.com")
print(f"  2. Navigate to MyDrive/RelCheck_Data/")
